# Automatic generation of Notebook using PyCropML
    This notebook implements a crop model.

### Model Referenceevapotranspiration

In [ ]:
MODULE Referenceevapotranspirationmod
    IMPLICIT NONE
CONTAINS

    SUBROUTINE model_referenceevapotranspiration(height_nn, &
        latent_heat_of_vaporization, &
        mean_air_temperature, &
        wind_speed, &
        wind_speed_height, &
        net_radiation, &
        stomata_resistance, &
        saturation_vapor_pressure_deficit, &
        saturated_vapor_pressure_slope, &
        psychrometric_constant, &
        reference_evapotranspiration)
        IMPLICIT NONE
        INTEGER:: i_cyml_r
        REAL, INTENT(IN) :: height_nn
        REAL, INTENT(IN) :: latent_heat_of_vaporization
        REAL, INTENT(IN) :: mean_air_temperature
        REAL, INTENT(IN) :: wind_speed
        REAL, INTENT(IN) :: wind_speed_height
        REAL, INTENT(IN) :: net_radiation
        REAL, INTENT(IN) :: stomata_resistance
        REAL, INTENT(IN) :: saturation_vapor_pressure_deficit
        REAL, INTENT(IN) :: saturated_vapor_pressure_slope
        REAL, INTENT(IN) :: psychrometric_constant
        REAL, INTENT(OUT) :: reference_evapotranspiration
        REAL:: atmospheric_pressure
        REAL:: psycrometer_constant
        REAL:: saturated_vapour_pressure_slope
        REAL:: inverse_lambda
        REAL:: wind_speed_2m
        REAL:: aerodynamic_resistance
        REAL:: surface_resistance
        !- Name: ReferenceEvapotranspiration -Version: 1, -Time step: 1
        !- Description:
    !            * Title: MONICA reference evapotranspiration
    !            * Authors: Claas Nendel (transcription into Crop2ML by Michael Berg-Mohnicke)
    !            * Reference: 
    !            Allen RG, Pereira LS, Raes D, Smith M. (1998) Crop evapotranspiration. Guidelines for computing crop water requirements. FAO Irrigation and
    !            Drainage Paper 56, FAO, Roma
    !        
    !            * Institution: ZALF e.V.
    !            * ExtendedDescription: 
    !            A method following Penman-Monteith as described by the FAO in Allen
    !            RG, Pereira LS, Raes D, Smith M. (1998) Crop evapotranspiration.
    !            Guidelines for computing crop water requirements. FAO Irrigation and
    !            Drainage Paper 56, FAO, Roma
    !        
    !            * ShortDescription: Calculates the MONICA reference evapotranspiration
        !- inputs:
    !            * name: height_nn
    !                          ** description : height above sea level
    !                          ** inputtype : parameter
    !                          ** parametercategory : constant
    !                          ** datatype : DOUBLE
    !                          ** max : 9999
    !                          ** min : -9999
    !                          ** default : 0
    !                          ** unit : m
    !            * name: latent_heat_of_vaporization
    !                          ** description : latent heat of vaporization lambda (0: MONICA's constants, 0.408 = 1/2.45 and psychrometric constant 0.000665 * pressure)
    !                          ** inputtype : parameter
    !                          ** parametercategory : constant
    !                          ** datatype : DOUBLE
    !                          ** max : 10
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : MJ/kg
    !            * name: mean_air_temperature
    !                          ** description : daily average air temperature
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 100
    !                          ** min : -100
    !                          ** default : 0
    !                          ** unit : °C
    !            * name: wind_speed
    !                          ** description : wind speed measured at wind speed height
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 9999
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : m/s
    !            * name: wind_speed_height
    !                          ** description : height at which the wind speed has been measured
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 9999
    !                          ** min : 0.1
    !                          ** default : 2
    !                          ** unit : m
    !            * name: net_radiation
    !                          ** description : net radiation
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : MJ/m2
    !            * name: stomata_resistance
    !                          ** description : stomata resistance (FAO default value is 100)
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 10000
    !                          ** min : 0
    !                          ** default : 100
    !                          ** unit : s/m
    !            * name: saturation_vapor_pressure_deficit
    !                          ** description : saturation_vapor_pressure_deficit
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : kPa
    !            * name: saturated_vapor_pressure_slope
    !                          ** description : slope of the saturation vapour pressure curve (negative: calculated from the mean air temperature)
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : -1
    !                          ** default : -1
    !                          ** unit : kPa/°C
    !            * name: psychrometric_constant
    !                          ** description : psychrometric constant (negative: calculated from height_nn and latent_heat_of_vaporization)
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : -1
    !                          ** default : -1
    !                          ** unit : kPa/°C
        !- outputs:
    !            * name: reference_evapotranspiration
    !                          ** description : reference evapotranspiration (ET0)
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** unit : mm
        atmospheric_pressure = 101.3 *  (((293.0 - (0.0065 * height_nn)) /  &
                293.0) ** 5.26)
        IF(psychrometric_constant .GE. 0.0) THEN
            psycrometer_constant = psychrometric_constant
        ELSE IF ( latent_heat_of_vaporization .GT. 0.0) THEN
            psycrometer_constant = 1.013e-3 * atmospheric_pressure / (0.622 *  &
                    latent_heat_of_vaporization)
        ELSE
            psycrometer_constant = 0.000665 * atmospheric_pressure
        END IF
        IF(saturated_vapor_pressure_slope .GE. 0.0) THEN
            saturated_vapour_pressure_slope = saturated_vapor_pressure_slope
        ELSE
            saturated_vapour_pressure_slope = 4098.0 * (0.6108 * EXP(17.27 *  &
                    mean_air_temperature / (mean_air_temperature + 237.3))) /  &
                    ((mean_air_temperature + 237.3) * (mean_air_temperature + 237.3))
        END IF
        IF(latent_heat_of_vaporization .GT. 0.0) THEN
            inverse_lambda = 1.0 / latent_heat_of_vaporization
        ELSE
            inverse_lambda = 0.408
        END IF
        wind_speed_2m = MAX(0.5, wind_speed * (4.87 / LOG((67.8 *  &
                wind_speed_height - 5.42))))
        aerodynamic_resistance = 208.0 / wind_speed_2m
        surface_resistance = stomata_resistance / 1.44
        reference_evapotranspiration = (inverse_lambda *  &
                saturated_vapour_pressure_slope * net_radiation +  &
                (psycrometer_constant * (900.0 / (mean_air_temperature + 273.0)) *  &
                wind_speed_2m * saturation_vapor_pressure_deficit)) /  &
                (saturated_vapour_pressure_slope + (psycrometer_constant * (1.0 +  &
                (surface_resistance / aerodynamic_resistance))))
        IF(reference_evapotranspiration .LT. 0.0) THEN
            reference_evapotranspiration = 0.0
        END IF
    END SUBROUTINE model_referenceevapotranspiration

END MODULE
PROGRAM test
    USE Referenceevapotranspirationmod

END PROGRAM
